In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp


path = "/Users/kausaliavijayaragavan/Documents/NG_scRNA/adata_all_donors_all_cell_states_raw_counts_in_raw_normlog_counts_in_X_for_download_UPD_20230307.h5ad"

adata = sc.read_h5ad(path)
adata = adata.copy()

adata.obs.columns
adata.obs["cell_type"].value_counts()  
group_key = "cell_type"

In [ ]:
#PCA trophoblast cluster
#adata_sub_troph = adata[adata.obs["coarse_annot"] == "Trophoblast"].copy()
keep = ["EVT_1", "EVT_2", "iEVT", "VCT", "GC"]
adata_sub_EVT = adata[adata.obs["cell_type"].isin(keep)].copy()

In [3]:

sc.pp.highly_variable_genes(adata_sub_EVT, n_top_genes=2000, flavor="cell_ranger")
sc.tl.pca(adata_sub_EVT) 

sc.pl.pca_variance_ratio(adata_sub_EVT, n_pcs=50, log=True)

In [ ]:
#check for dataset separation
sc.pl.pca(adata_sub_EVT, color="dataset", dimensions=[(0,1),(2,3)], ncols=1, size=2)

In [ ]:
#find cell type separation
sc.pl.pca(
    adata_sub_EVT,
    color=["cell_type", "cell_type", "cell_type"],
    dimensions=[(0, 1), (2, 3),(4,5)],
    ncols=1,
    size=2,
)

In [ ]:
#use PC2 to find top 400 genes that are responsible for 2nd largest variation (biological)
def top_pc_genes(adata, pc, n_genes=400, save_prefix=None):
    """
    Return top genes by absolute loading for PC `pc` (1-based like R: PC1=1, PC2=2).
    Requires PCA already run: sc.tl.pca(adata, ...)
    """
    pc0 = pc - 1  # to 0-based
    load = pd.Series(adata.varm["PCs"][:, pc0], index=adata.var_names)  # loadings

    top = load.abs().sort_values(ascending=False).head(n_genes).index.tolist()
    top_pos = load.loc[top][load.loc[top] > 0].index.tolist()
    top_neg = load.loc[top][load.loc[top] < 0].index.tolist()

    if save_prefix is not None:
        pd.Series(top).to_csv(f"{save_prefix}_PC{pc}_top{n_genes}_all.txt", index=False, header=False)
        pd.Series(top_pos).to_csv(f"{save_prefix}_PC{pc}_top{n_genes}_pos.txt", index=False, header=False)
        pd.Series(top_neg).to_csv(f"{save_prefix}_PC{pc}_top{n_genes}_neg.txt", index=False, header=False)

    return top, top_pos, top_neg

def pc_gene_heatmap(adata, genes, groupby="leiden", layer=None, standard_scale="var", **kwargs):
    """
    Quick Scanpy heatmap of genes (group means). Uses Scanpy scaling (0-1 if standard_scale='var').
    """
    sc.pl.heatmap(
        adata,
        var_names=[g for g in genes if g in adata.var_names],
        groupby=groupby,
        layer=layer,
        standard_scale=standard_scale,
        swap_axes=True,
        **kwargs,
    )

In [ ]:
#validate marker expression across cells
import matplotlib.pyplot as plt
import seaborn as sns

pc2 = adata_sub_EVT.obsm["X_pca"][:, 1]  # PC2 (0-based index)
hlag = adata_sub_EVT[:, "HLA-G"].X

order = ["VCT", "EVT_1", "EVT_2", "iEVT", "GC"]

# make it 1D array
hlag = hlag.toarray().ravel() if hasattr(hlag, "toarray") else np.array(hlag).ravel()

df = pd.DataFrame({
    "PC2": pc2,
    "HLA-G": hlag,
    "cell_type": adata_sub_EVT.obs["cell_type"].astype(str).values
})

plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x="PC2", y="HLA-G", hue="cell_type", s=12, hue_order=order, linewidth=0)
plt.tight_layout()
plt.show()

In [ ]:
#more markers
genes = ["HLA-G","PRG2","AOC1","PAPPA2","NOTCH1","ERBB2","NOTUM","PTGS2"]
order = ["VCT", "EVT_1", "EVT_2", "iEVT", "GC"]

pc2 = adata_sub_EVT.obsm["X_pca"][:, 1]

# extract expression
X = adata_sub_EVT[:, genes].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

df = pd.DataFrame(X, columns=genes)
df["PC2"] = pc2
df["cell_type"] = adata_sub_EVT.obs["cell_type"].astype(str).values

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)
axes = axes.ravel()

for i, (ax, g) in enumerate(zip(axes, genes)):
    sns.scatterplot(
        df, x="PC2", y=g,
        hue="cell_type", hue_order=order,
        s=10, linewidth=0, alpha=0.7,
        ax=ax,
        legend=(i == 0)
    )
    ax.set_title(g)
    ax.set_xlabel("PC2")
    ax.set_ylabel("Expr")
    if i != 0 and ax.legend_ is not None:
        ax.legend_.remove()

plt.tight_layout()
plt.show()

In [ ]:
#which genes in PC2?
# use functions on EVT subset

top, top_pos, top_neg = top_pc_genes(adata_sub_EVT, pc=2, n_genes=400, save_prefix="EVTsubset")

order = ["VCT", "EVT_1", "EVT_2", "iEVT", "GC"]

adata_sub_EVT.obs["cell_type"] = adata_sub_EVT.obs["cell_type"].astype("category")
adata_sub_EVT.obs["cell_type"] = adata_sub_EVT.obs["cell_type"].cat.set_categories(order, ordered=True)

sc.pl.dotplot(
    adata_sub_EVT,
    var_names=top_pos[:20],
    groupby="cell_type",
    standard_scale="var",   # scales per gene across groups (like your matrixplot)
    swap_axes=True,
    cmap="RdBu_r",
    dot_max=0.6 # optional: cap dot size
)

In [51]:
#ORA plots using R

import pandas as pd
import subprocess, textwrap, pathlib

#ORA (clusterProfiler) from Python via Rscript
import pandas as pd, subprocess, textwrap, pathlib

genes = top_pos  # your SYMBOL list
outdir = pathlib.Path("ora_clusterprofiler_PC2_pos_ALL")
outdir.mkdir(parents=True, exist_ok=True)

genes_file = outdir / "genes.txt"
pd.Series(genes).to_csv(genes_file, index=False, header=False)

r_script = outdir / "run_ora_ALL.R"
r_script.write_text(textwrap.dedent(f"""
suppressPackageStartupMessages({{
  library(clusterProfiler)
  library(enrichplot)
  library(org.Hs.eg.db)
  library(ggplot2)
  library(stringr)
  
  if (!requireNamespace("msigdbr", quietly=TRUE))
  install.packages("msigdbr", repos="https://cloud.r-project.org")

  library(msigdbr)
}})

genes <- readLines("{genes_file}")

run_msig <- function(collection, subcollection=NULL, label, show=15) {{
  if (is.null(subcollection)) {{
    m <- msigdbr(species="Homo sapiens", collection=collection)
  }} else {{
    m <- msigdbr(species="Homo sapiens", collection=collection, subcollection=subcollection)
  }}

  term2gene <- m[, c("gs_name", "gene_symbol")]
  colnames(term2gene) <- c("term", "gene")

  enr <- enricher(gene=genes, TERM2GENE=term2gene,
                  pAdjustMethod="BH", pvalueCutoff=0.05, qvalueCutoff=0.05)

  out_csv <- file.path("{outdir}", paste0(label, "_results.csv"))
  write.csv(as.data.frame(enr), out_csv, row.names=FALSE)

  if (is.null(enr) || nrow(as.data.frame(enr)) == 0) {{
    message("No hits for: ", label)
    return(invisible(NULL))
  }}

  p1 <- dotplot(enr, showCategory=show) +
    ggtitle(paste0(label, " ORA")) +
    scale_y_discrete(labels=function(x) str_wrap(x, width=40)) +
    theme(axis.text.y = element_text(size=8))
  ggsave(file.path("{outdir}", paste0(label, "_dotplot.png")), p1, width=10, height=9, dpi=200)

  p2 <- barplot(enr, showCategory=show) + ggtitle(paste0(label, " ORA"))
  ggsave(file.path("{outdir}", paste0(label, "_barplot.png")), p2, width=10, height=7, dpi=200)
}}

run_msig("H",  label="H_Hallmark")
run_msig("C6", label="C6_Oncogenic")
run_msig("C2", subcollection="CGP",          label="C2_CGP")
run_msig("C2", subcollection="CP:REACTOME",  label="C2_CP_REACTOME")
run_msig("C7", label="C7_Immunologic")
run_msig("C8", label="C8_CellType")
"""))

subprocess.run(["Rscript", str(r_script)], check=True)
print("Done. Outputs in:", outdir)
#subprocess.run(["Rscript", str(r_script)], check=True)

Warning messages:
1: package ‘ggplot2’ was built under R version 4.4.3 
2: package ‘msigdbr’ was built under R version 4.4.3 
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Warning messages:
1: In fortify(object, showCategory = showCategory, by = x, ...) :
  Arguments in `...` must be used.
✖ Problematic argument:
• by = x
ℹ Did you misspell an argument name?
2: `aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the enrichplot package.
  Please report the issue at
  <https://github.com/GuangchuangYu/enrichplot/issues>. 
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Warning message:
In fortify(object, showCategory = showCategory, by = x, ...) :
  Arguments in `...` must be used.
✖ Problematic argument:
• by = x
ℹ Did you misspel

Done. Outputs in: ora_clusterprofiler_PC2_pos_ALL


Warning message:
In fortify(object, showCategory = showCategory, by = x, ...) :
  Arguments in `...` must be used.
✖ Problematic argument:
• by = x
ℹ Did you misspell an argument name?
